# Build APK de English Quiz

Ejecuta estas celdas en orden. La primera compilación puede tardar varios minutos porque Google Colab descarga Android SDK y NDK.

In [ ]:
import os

REPO_URL = "https://github.com/Ardelvi86/english-quiz.git"
PROJECT_DIR = "/content/english-quiz"

if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

In [ ]:
!sudo apt-get update
!sudo apt-get install -y git zip unzip openjdk-17-jdk python3-pip python3-dev build-essential autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo6 cmake libffi-dev libssl-dev automake

In [ ]:
%pip install --upgrade pip setuptools wheel buildozer cython==0.29.34

In [ ]:
import os
import subprocess
import time

env = os.environ.copy()
env["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
log_path = "build.log"
started = time.monotonic()

with open(log_path, "w", encoding="utf-8") as log:
    process = subprocess.Popen(
        ["buildozer", "android", "debug"],
        stdout=log,
        stderr=subprocess.STDOUT,
        env=env,
    )
    while process.poll() is None:
        time.sleep(30)
        log.flush()
        with open(log_path, "rb") as output:
            output.seek(max(0, os.path.getsize(log_path) - 4096))
            recent = output.read().decode("utf-8", errors="replace").splitlines()
        progress = recent[-1] if recent else "Preparando la compilación..."
        print(f"{(time.monotonic() - started) / 60:.1f} min | {progress[-250:]}")

if process.returncode != 0:
    with open(log_path, "r", encoding="utf-8", errors="replace") as log:
        lines = log.readlines()
    print("\nÚltimas líneas del registro:\n" + "".join(lines[-80:]))
    raise RuntimeError(f"Buildozer terminó con código {process.returncode}; revisa build.log.")

print("Compilación completada.")

In [ ]:
import glob
from google.colab import files

apks = glob.glob("bin/*.apk")
print("APKs encontrados:", apks)

for apk in apks:
    files.download(apk)